In [4]:
import pandas as pd
import numpy as np


In [17]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 120)
    
RAW_DIR  = '../student_resource/dataset/train/' 
PROC_DIR = '../student_resource/processed'

In [18]:
import os
os.makedirs(PROC_DIR, exist_ok=True)

#### Loading all files

In [19]:
s1 = pd.read_csv(f'{RAW_DIR}/train_source1.tsv', sep='\t')
s2 = pd.read_csv(f'{RAW_DIR}/train_source2.tsv', sep='\t')
s3 = pd.read_csv(f'{RAW_DIR}/train_source3.tsv', sep='\t')
gt  = pd.read_csv(f'{RAW_DIR}/train_ground_truth.tsv', sep='\t')

print(f'Source 1 (reference): {len(s1):,} rows')
print(f'Source 2 (noisy)    : {len(s2):,} rows')
print(f'Ground truth        : {len(gt):,} rows')
s1.head(3)

Source 1 (reference): 2,206,821 rows
Source 2 (noisy)    : 5,034,616 rows
Ground truth        : 2,206,821 rows


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US


In [20]:
for name, df in [('S1', s1), ('S2', s2), ('S3', s3)]:
    print(f'{name}')
    display(df.head(2))
    print(df.dtypes.to_string())
    print()

S1


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US


entity_id           str
business_name       str
business_address    str
country             str

S2


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US


entity_id           str
business_name       str
business_address    str
country             str

S3


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India


entity_id           str
business_name       str
business_address    str
country             str



In [ ]:
## missing values 
for i in [1, 2, 3]:
    df = pd.read_csv(f"../student_resource/dataset/train/train_source{i}.tsv", sep="\t")
    print(f"source{i}: {len(df)} rows, cols={df.columns.tolist()}")
    print(df.isna().mean().round(3))

source1: 2206821 rows, cols=['entity_id', 'business_name', 'business_address', 'country']
entity_id           0.0
business_name       0.0
business_address    0.0
country             0.0
dtype: float64
source2: 5034616 rows, cols=['entity_id', 'business_name', 'business_address', 'country']
entity_id           0.000
business_name       0.000
business_address    0.034
country             0.000
dtype: float64
source3: 5285603 rows, cols=['entity_id', 'business_name', 'business_address', 'country']
entity_id           0.000
business_name       0.000
business_address    0.033
country             0.000
dtype: float64


### train_ground_truth file

In [ ]:
gt = pd.read_csv("../student_resource/dataset/train/train_ground_truth.tsv", sep="\t")
print(gt.shape)
print(gt.columns.tolist())
gt.head(5)

(2206821, 2)
['source1_entity_id', 'matched_entity_ids']


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762..."
6,S1-318373630,"S2-660036492,S3-804600254"
7,S1-86989137,"S3-274817120,S3-312496301"
8,S1-29845983,"S2-648035184,S3-588502663"
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439"


In [7]:
gt["n_matches"] = gt["matched_entity_ids"].fillna("").apply(
    lambda s: 0 if s == "" else len(s.split(",")))
print("S1 entities:", len(gt))

S1 entities: 2206821


In [8]:
print(gt["n_matches"].describe())

count    2.206821e+06
mean     3.461253e+00
std      1.705323e+00
min      0.000000e+00
25%      2.000000e+00
50%      3.000000e+00
75%      5.000000e+00
max      1.100000e+01
Name: n_matches, dtype: float64


In [ ]:
## no matches
print("singleton fraction:", (gt["n_matches"] == 0).mean().round(3))   # critical number!

singleton fraction: 0.056


In [10]:
print("match count distribution:\n", gt["n_matches"].value_counts().sort_index().head(10))

match count distribution:
 n_matches
0    123247
1    119157
2    375212
3    530841
4    484115
5    321957
6    164868
7     63968
8     18680
9      4205
Name: count, dtype: int64
